# Sera-Z — Latih Model 1 (YOLOv11n-cls) di Colab

Notebook ini sudah final. Jalankan sel dari atas ke bawah, jangan melompat.

- Runtime: **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- Repo private: siapkan GitHub token, atau upload `ml.zip`
- Waktu: persiapan data 10-20 menit, training 30 epoch sekitar 1,5-2 jam
- Jangan tutup tab selama training
- Hasil disimpan ke Google Drive di sel kedua dari akhir

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

Mon Nov  9 14:56:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Ambil kode

Kalau folder repo belum ada, sel ini akan clone. Untuk repo private, tempel
GitHub token saat diminta (token tidak ditampilkan). Kalau lebih suka tanpa
token, upload `ml.zip` lalu jalankan sel unzip di bawah.

In [6]:
import os

DIR = None
for d in ["/content/sera-z", "/content/PBL-Mobile-Kelompok2-Semester5", "/content"]:
    if os.path.isdir(f"{d}/ml/scripts"):
        DIR = d
        break

if DIR is None:
    print("Folder repo belum ada. Upload ml.zip saat diminta.")
    !mkdir -p /content/sera-z
    from google.colab import files
    files.upload()
    !unzip -q ml.zip -d /content/sera-z
    DIR = "/content/sera-z"

os.chdir(DIR)
print("DIR:", DIR)
print("cwd:", os.getcwd())
print("script:", sorted(os.listdir("ml/scripts")))

Folder repo belum ada. Upload ml.zip saat diminta.


Saving ml.zip to ml.zip
DIR: /content/sera-z
cwd: /content/sera-z
script: ['00_download_data.py', '01_prepare_model1.py', '02_balance_train.py', '03_baseline_mobilenetv2.py', '04_train_yolo11n.py', '05_eval_model1.py', '06_export_tflite.py', '07_extract_color_kmeans.py', '08_train_model3_mlp.py', '09_prepare_hf.py', '10_eval_field.py', '11_download_negatives.py', '12_run_model1_pipeline.py']


Alternatif tanpa Git: upload `ml.zip` (dibuat dengan perintah di bawah), lalu
jalankan sel unzip ini.

```powershell
git archive --format=zip -o D:\Temp\ml.zip ml/model1-v2-pipeline ml
```

In [ ]:
# Jalankan hanya jika memakai ml.zip. Lewati kalau clone berhasil.
# !mkdir -p /content/sera-z
# from google.colab import files
# files.upload()
# !unzip -q ml.zip -d /content/sera-z
# import os; os.chdir("/content/sera-z")
# print(os.getcwd(), sorted(os.listdir("ml/scripts")))

## 2. Pasang dependensi

In [7]:
!pip install -q ultralytics datasets pillow pandas scikit-learn matplotlib tqdm
import ultralytics, datasets, torch
print("ultralytics:", ultralytics.__version__)
print("datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics: 8.4.208
datasets: 4.8.5
CUDA: True


## 3. Pastikan script COCO sudah dipatch

Patch ini membuat script membaca kategori COCO di berbagai versi `datasets`.
Aman dijalankan berulang; kalau sudah ada, tidak diubah lagi.

In [8]:
from pathlib import Path

p = Path("ml/label_maps/fashion_products.csv")
t = p.read_text()
t = t.replace("Sweatshirts,outerwear", "Sweatshirts,long_sleeve")
t = t.replace("Sweaters,outerwear", "Sweaters,long_sleeve")
p.write_text(t)
print(t)

source_label,target
Tshirts,tshirt
Tops,tshirt
Shirts,shirt
Sweatshirts,long_sleeve
Sweaters,long_sleeve
Jackets,outerwear
Blazers,outerwear
Rain Jacket,outerwear
Trousers,long_pants
Jeans,long_pants
Track Pants,long_pants
Jeggings,long_pants
Shorts,shorts
Skirts,skirt
Dresses,dress
Sandals,shoes
Casual Shoes,shoes
Formal Shoes,shoes
Flip Flops,shoes
Heels,shoes
Sports Shoes,shoes



## 4. Unduh 800 foto objek bukan pakaian

Kelas `other` berisi laptop, meja, kursi, orang, botol, TV, buku, ruangan.
Dilewati otomatis kalau folder sudah terisi.

In [10]:
from pathlib import Path

p = Path("ml/scripts/11_download_negatives.py")
if not p.exists():
    print("11_download_negatives.py tidak ditemukan. Upload ulang ml.zip terbaru.")
else:
    kode = p.read_text()
    if "kandidat" in kode:
        print("Patch sudah ada.")
    else:
        mulai = kode.index("def nama_kategori")
        akhir = kode.index("def main()")
        fungsi_baru = '''def nama_kategori(ds, row) -> set[str] | None:
    raw = row.get("objects", {}).get("category", [])
    if not raw:
        return set()
    if all(isinstance(c, str) for c in raw):
        return set(raw)

    fitur = ds.features["objects"]
    kandidat = None
    if hasattr(fitur, "feature"):
        dalam = fitur.feature
        kandidat = dalam.get("category") if isinstance(dalam, dict) else None
    elif isinstance(fitur, dict):
        kandidat = fitur.get("category")
        if kandidat is None and isinstance(fitur.get("feature"), dict):
            kandidat = fitur["feature"].get("category")

    names = getattr(kandidat, "names", None)
    if not names:
        return None
    return {names[c] if isinstance(c, int) else c for c in raw}


'''
        kode = kode[:mulai] + fungsi_baru + kode[akhir:]
        kode = kode.replace(
            "if args.prefer_indoor and not (kategori & KATEGORI_DALAM_RUANG):",
            "if args.prefer_indoor and kategori is not None and not (kategori & KATEGORI_DALAM_RUANG):",
        )
        p.write_text(kode)
        print("Patch diterapkan.")


Patch diterapkan.


In [11]:
from pathlib import Path

NEG = Path("ml/data/raw/negatives")
jumlah = len(list(NEG.rglob("*.jpg")))
print("Foto negatif:", jumlah)

if jumlah < 700:
    !python ml/scripts/11_download_negatives.py --count 800
else:
    print("Sudah cukup, unduh dilewati.")

Foto negatif: 0
Memuat detection-datasets/coco (streaming, tidak mengunduh seluruh dataset)...
README.md: 100% 58.0/58.0 [00:00<00:00, 266kB/s]
Resolving data files: 100% 40/40 [00:00<00:00, 10224.40it/s]
dataset_infos.json: 100% 2.10k/2.10k [00:00<00:00, 5.42MB/s]
mengunduh: 100% 800/800 [00:12<00:00, 64.71it/s] 

Selesai: 800 foto disimpan di /content/sera-z/ml/data/raw/negatives
1 foto dilewati (tidak sesuai filter).

Lanjutkan dengan:
  python ml/scripts/09_prepare_hf.py --force --negatives "/content/sera-z/ml/data/raw/negatives"


## 5. Siapkan dataset 10 kelas

Perkiraan 10-20 menit. Tanda selesai: `metadata.csv ditulis`.

In [12]:
from pathlib import Path

prep = Path("ml/scripts/13_prepare_hf.py")
if not prep.exists():
    prep = Path("ml/scripts/09_prepare_hf.py")
print("Memakai:", prep)

!python {prep} --force --negatives ml/data/raw/negatives

Memakai: ml/scripts/09_prepare_hf.py
Mengunduh metadata ashraq/fashion-product-images-small (parquet ~270 MB pada run pertama)...
README.md: 100% 867/867 [00:00<00:00, 4.76MB/s]

data/train-00000-of-00002-6cff4c59f91661(…): downloading bytes:  70% 94.7M/136M [00:01<00:00, 125MB/s, 5.49MB/s  ] 
data/train-00000-of-00002-6cff4c59f91661(…): reconstructing file:  49% 67.1M/136M [00:01<00:01, 47.8MB/s]
data/train-00000-of-00002-6cff4c59f91661(…): downloading bytes: 100% 117M/117M [00:01<00:00, 74.5MB/s, 11.2MB/s  ]
data/train-00000-of-00002-6cff4c59f91661(…): reconstructing file: 100% 136M/136M [00:01<00:00, 86.5MB/s, 13.1MB/s  ]

data/train-00001-of-00002-bb459e5ac5f01e(…): downloading bytes:  80% 108M/135M [00:01<00:00, 172MB/s, 6.81MB/s  ] 
data/train-00001-of-00002-bb459e5ac5f01e(…): downloading bytes: 100% 117M/117M [00:01<00:00, 94.8MB/s, 11.2MB/s  ]
data/train-00001-of-00002-bb459e5ac5f01e(…): reconstructing file: 100% 135M/135M [00:01<00:00, 110MB/s, 13.1MB/s  ]
Generating train spl

## 6. Verifikasi dataset

Harus ada 10 kelas termasuk `other`, dan semua split terisi.

In [13]:
import pandas as pd

m = pd.read_csv("ml/data/processed/model1/metadata.csv")
print(m.groupby(["split", "target"]).size().unstack(fill_value=0).to_string())
print()
print("Kelas:", sorted(m["target"].unique()))
print("Jumlah kelas:", m["target"].nunique())

target  dress  long_pants  long_sleeve  other  outerwear  shirt  shoes  shorts  skirt  tshirt
split                                                                                        
test       57         212           84    120         40    407   1275      66     15    1184
train     272         992          389    560        185   2067   5956     312     71    5549
val        57         213           84    120         40    407   1277      63     12    1182

Kelas: ['dress', 'long_pants', 'long_sleeve', 'other', 'outerwear', 'shirt', 'shoes', 'shorts', 'skirt', 'tshirt']
Jumlah kelas: 10


## 7. Latih model

30 epoch, batch 64, 224 px. Perkiraan 1,5-2 jam di T4.

In [14]:
!python ml/scripts/04_train_yolo11n.py \
    --data ml/data/processed/model1 \
    --epochs 30 --imgsz 224 --batch 64 \
    --workers 2 --device 0 --name model1_hf_v3

Kelas (10): ['dress', 'long_pants', 'long_sleeve', 'other', 'outerwear', 'shirt', 'shoes', 'shorts', 'skirt', 'tshirt']
Train: 16353 gambar | Val: 3455 gambar

Mulai latih: yolo11n-cls.pt | epochs=30 | imgsz=224

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ml/data/processed/model1, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=None, kobj=1.0, line_w

## 8. Evaluasi test katalog

In [15]:
!python ml/scripts/05_eval_model1.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v3/weights/best.pt

import json
d = json.load(open("ml/reports/model1_test_summary.json"))
print("Akurasi:", d["accuracy"], "| Macro F1:", d["macro_f1"])

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: /content/sera-z/ml/data/processed/model1/train... found 16353 images in 10/10 classes ✅ 
val: /content/sera-z/ml/data/processed/model1/val... found 3455 images in 10/10 classes ✅ 
test: /content/sera-z/ml/data/processed/model1/test... found 3460 images in 10/10 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 84.8±40.3 MB/s, size: 2.3 KB)
test: Scanning /content/sera-z/ml/data/processed/model1/test... 3460 images, 0 corrupt: 100% ━━━━━━━━━━━━ 3460/3460 6.1Kit/s 0.6s
test: New cache created: /content/sera-z/ml/data/processed/model1/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 217/217 26.7it/s 8.1s
                   all      0.969          1
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/sera-z/runs/classify/va

## 8b. Validasi ambang penolakan

Cari ambang confidence yang menolak objek bukan pakaian tanpa banyak salah menolak pakaian.


In [16]:
from pathlib import Path

sweep = Path("ml/scripts/14_threshold_sweep.py")
if sweep.exists():
    !python {sweep} --weights ml/runs/model1_hf_v3/weights/best.pt
else:
    print("14_threshold_sweep.py tidak ada di sesi ini, dilewati.")

14_threshold_sweep.py tidak ada di sesi ini, dilewati.


## 9. Ekspor TFLite int8

Pastikan output verifikasi tertulis `[1, 10]`.

In [17]:
!pip install -q tensorflow onnx onnxslim onnxruntime

!python ml/scripts/06_export_tflite.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v3/weights/best.pt \
    --out-name model1_yolo11n_v3.tflite

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from 'ml/runs/model1_hf_v3/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 10) (3.1 MB)
LiteRT: collecting INT8 calibration images from 'data=ml/data/processed/model1'
train: /content/sera-z/ml/data/processed/model1/train... found 16353 images in 10/10 classes ✅ 
val: /content/sera-z/ml/data/processed/model1/val... found 3455 images in 10/10

## 10. Simpan ke Google Drive

Wajib sebelum sesi Colab berakhir.

In [18]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/SeraZ-ML"
!cp -r ml/runs/model1_hf_v3 "/content/drive/MyDrive/SeraZ-ML/"
!cp ml/reports/model1_test_* "/content/drive/MyDrive/SeraZ-ML/"
!cp assets/models/model1_yolo11n_v3.tflite "/content/drive/MyDrive/SeraZ-ML/"

import os
print(os.listdir("/content/drive/MyDrive/SeraZ-ML"))
!zip -r -q /content/model1_v3.zip \
    ml/runs/model1_hf_v3 \
    ml/reports \
    assets/models/model1_yolo11n_v3.tflite

from google.colab import files
files.download("/content/model1_v3.zip")
files.download("/content/sera-z/assets/models/model1_yolo11n_v3.tflite")

Mounted at /content/drive
['model1_hf_v2', 'model1_test_confusion_matrix.png', 'model1_test_per_source.csv', 'model1_test_report.txt', 'model1_test_summary.json', 'model1_yolo11n_v2.tflite', 'model1_hf_v3', 'model1_yolo11n_v3.tflite']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import tensorflow as tf

it = tf.lite.Interpreter(model_path="assets/models/model1_yolo11n_v3.tflite")
it.allocate_tensors()
print("Input :", it.get_input_details()[0]["shape"])
print("Output:", it.get_output_details()[0]["shape"])

Input : [  1   3 224 224]
Output: [ 1 10]


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 11. Unduh ke laptop

In [20]:
!zip -r -q /content/model1_v3.zip \
    ml/runs/model1_hf_v3 \
    ml/reports \
    assets/models/model1_yolo11n_v3.tflite

from google.colab import files
files.download("/content/model1_v3.zip")
files.download("/content/sera-z/assets/models/model1_yolo11n_v3.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
!zip -r -q /content/model1_v2.zip ml/runs/model1_hf_v2 ml/reports assets/models/model1_yolo11n_v2.tflite
from google.colab import files
files.download("/content/model1_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>